# Langham Hotel Management System



## 1. Setup and Shared Data

The system keeps room records and active allocations in two list-based collections.  
Using lists of dictionaries makes the output easy to read in a notebook and keeps the structure clear for a beginner-level console program.

In [1]:
"""
File        : BBIM502_A2_Project_850007583_Alternative.ipynb
Author      : Student 850008628
Course      : BBIM502 Introduction to Programming
Description : Console-based Langham Hotel Management System using Python.
Date        : 2026

This program demonstrates:
- Room adding and deleting
- Room display before booking
- Room allocation
- Allocation display
- Billing and room de-allocation
- File save, file display, backup, and clear operations
- Exception handling for common Python errors
"""

from pathlib import Path
from datetime import datetime
import math
import os

STUDENT_ID = "850008628"
ALLOCATION_FILE = Path(f"LHMS_{STUDENT_ID}.txt")

# Room records are stored as dictionaries inside this list.
# Example: {"room_no": "101", "category": "Deluxe", "nightly_rate": 180.0, "features": "WiFi, TV", "status": "Available"}
rooms = []

# Active bookings are stored separately from the room list.
# Example: {"booking_id": "BK101...", "room_no": "101", "guest_name": "John", "phone": "021...", "nights": 2, "check_in": "..."}
room_allocations = []

print("Langham Hotel Management System is ready.")
print("Allocation file:", ALLOCATION_FILE)

Langham Hotel Management System is ready.
Allocation file: LHMS_850008628.txt


## 2. General Helper Functions

These functions reduce repeated code.  
They help the rest of the program find rooms, validate room numbers, and keep the console output consistent.

In [2]:
def print_rule(symbol="-", length=70):
    """Prints a simple horizontal separator line."""
    print(symbol * length)


def normalise_room_number(room_no):
    """
    Converts a room number into a clean string.

    Raises:
        ValueError: if the room number is blank or contains non-numeric characters.
    """
    clean_room_no = str(room_no).strip()

    if clean_room_no == "":
        raise ValueError("Room number cannot be blank.")

    if not clean_room_no.isdigit():
        raise ValueError("Room number must contain digits only.")

    return clean_room_no


def find_room(room_no):
    """Searches the room list and returns the matching room dictionary, or None."""
    clean_room_no = normalise_room_number(room_no)

    for room in rooms:
        if room["room_no"] == clean_room_no:
            return room

    return None


def find_allocation(room_no):
    """Searches the active allocation list and returns the matching booking, or None."""
    clean_room_no = normalise_room_number(room_no)

    for allocation in room_allocations:
        if allocation["room_no"] == clean_room_no:
            return allocation

    return None


def read_integer(message, minimum=None):
    """
    Reads an integer from the keyboard and repeats until valid input is provided.

    Handles:
        ValueError: when the user types text instead of a number.
        EOFError: when input is unexpectedly stopped.
    """
    while True:
        try:
            value = int(input(message))

            if minimum is not None and value < minimum:
                print(f"Please enter a value of at least {minimum}.")
                continue

            return value

        except ValueError:
            print("Invalid input. Please enter a whole number.")
        except EOFError:
            print("Input ended unexpectedly.")
            raise


def read_decimal(message, minimum=None):
    """
    Reads a decimal number from the keyboard and repeats until valid input is provided.
    """
    while True:
        try:
            value = float(input(message))

            if minimum is not None and value < minimum:
                print(f"Please enter a value of at least {minimum}.")
                continue

            return value

        except ValueError:
            print("Invalid input. Please enter a valid number.")
        except EOFError:
            print("Input ended unexpectedly.")
            raise


def read_required_text(message):
    """Reads text and does not accept a blank answer."""
    while True:
        try:
            value = input(message).strip()

            if value == "":
                print("This field cannot be empty.")
                continue

            return value

        except EOFError:
            print("Input ended unexpectedly.")
            raise

## 3. Core Room and Booking Operations

The following functions perform the main logic of the system.  
They accept parameters, update the in-memory data, and return a message.  
This makes the functions easier to test in Jupyter before using the interactive menu.

### Option 1 — Add Room

In [3]:
def add_room_record(room_no, category, nightly_rate, features):
    """
    Adds one new room to the room list.

    Menu option covered:
        1. Add Room

    Exceptions handled:
        ValueError, TypeError
    """
    try:
        clean_room_no = normalise_room_number(room_no)

        if find_room(clean_room_no) is not None:
            return f"Room {clean_room_no} already exists. No duplicate room was added."

        room_category = str(category).strip()
        room_features = str(features).strip()
        rate = float(nightly_rate)

        if room_category == "":
            return "Room category cannot be empty."

        if rate <= 0:
            return "Nightly rate must be greater than zero."

        new_room = {
            "room_no": clean_room_no,
            "category": room_category,
            "nightly_rate": rate,
            "features": room_features if room_features else "Standard facilities",
            "status": "Available"
        }

        rooms.append(new_room)
        return f"Room {clean_room_no} has been added successfully."

    except ValueError as error:
        return f"Room could not be added. {error}"
    except TypeError as error:
        return f"Room could not be added due to incorrect data type: {error}"


def menu_add_room():
    """Keyboard-input wrapper for adding a room."""
    room_no = read_required_text("Enter room number: ")
    category = read_required_text("Enter room category/type: ")
    nightly_rate = read_decimal("Enter nightly rate: $", minimum=1)
    features = input("Enter room features: ").strip()

    print(add_room_record(room_no, category, nightly_rate, features))

### Option 2 — Delete Room

In [4]:
def delete_room_record(room_no):
    """
    Deletes a room if it exists and is not currently allocated.

    Menu option covered:
        2. Delete Room

    Logical protection:
        An occupied room cannot be deleted.
    """
    try:
        room = find_room(room_no)

        if room is None:
            return "Room was not found."

        if room["status"] == "Allocated":
            return "This room is currently allocated, so it cannot be deleted."

        rooms.remove(room)
        return f"Room {room['room_no']} has been deleted."

    except ValueError as error:
        return f"Room deletion failed. {error}"
    except KeyError as error:
        return f"Room deletion failed because the room record is missing this field: {error}"


def menu_delete_room():
    """Keyboard-input wrapper for deleting a room."""
    room_no = read_required_text("Enter room number to delete: ")
    print(delete_room_record(room_no))

### Option 3 — Display Room Details

In [5]:
def display_room_details():
    """
    Displays all room details before booking.

    Menu option covered:
        3. Display Room Details
    """
    try:
        if len(rooms) == 0:
            print("There are no rooms in the system yet.")
            return

        print("\nCurrent Room Details")
        print_rule("=")

        for room in sorted(rooms, key=lambda item: int(item["room_no"])):
            print(f"Room Number : {room['room_no']}")
            print(f"Type        : {room['category']}")
            print(f"Rate        : ${room['nightly_rate']:.2f} per night")
            print(f"Features    : {room['features']}")
            print(f"Status      : {room['status']}")
            print_rule()

    except ValueError:
        print("A room number is not numeric, so the list could not be sorted.")
    except KeyError as error:
        print(f"A room record is incomplete. Missing field: {error}")


def menu_display_rooms():
    """Keyboard-input wrapper for displaying rooms."""
    display_room_details()

### Option 4 — Allocate Room

In [6]:
def allocate_room_record(room_no, guest_name, phone, nights):
    """
    Allocates an available room to a customer.

    Menu option covered:
        4. Allocate Room
    """
    try:
        clean_room_no = normalise_room_number(room_no)
        room = find_room(clean_room_no)

        if room is None:
            return "Room was not found. Please add the room before allocation."

        if room["status"] != "Available":
            return "Room is not available for allocation."

        guest = str(guest_name).strip()
        contact_number = str(phone).strip()
        stay_nights = int(nights)

        if guest == "":
            return "Guest name cannot be empty."

        if contact_number == "":
            return "Phone number cannot be empty."

        if stay_nights <= 0:
            return "Number of nights must be greater than zero."

        booking = {
            "booking_id": f"BK-{clean_room_no}-{datetime.now().strftime('%Y%m%d%H%M%S')}",
            "room_no": clean_room_no,
            "guest_name": guest,
            "phone": contact_number,
            "nights": stay_nights,
            "check_in": datetime.now().strftime("%d/%m/%Y %H:%M")
        }

        room_allocations.append(booking)
        room["status"] = "Allocated"

        return f"Room {clean_room_no} has been allocated to {guest}."

    except ValueError as error:
        return f"Allocation failed. {error}"
    except TypeError as error:
        return f"Allocation failed due to incorrect data type: {error}"
    except NameError as error:
        return f"Allocation failed because a variable name is not defined: {error}"


def menu_allocate_room():
    """Keyboard-input wrapper for allocating a room."""
    room_no = read_required_text("Enter room number to allocate: ")
    guest_name = read_required_text("Enter guest name: ")
    phone = read_required_text("Enter guest phone number: ")
    nights = read_integer("Enter number of nights: ", minimum=1)

    print(allocate_room_record(room_no, guest_name, phone, nights))

### Option 5 — Display Room Allocation Details

In [7]:
def display_room_allocation_details():
    """
    Displays all active room allocations.

    Menu option covered:
        5. Display Room Allocation Details
    """
    try:
        if len(room_allocations) == 0:
            print("There are no active room allocations.")
            return

        print("\nActive Room Allocations")
        print_rule("=")

        for allocation in room_allocations:
            room = find_room(allocation["room_no"])
            rate = room["nightly_rate"] if room else 0

            print(f"Booking ID  : {allocation['booking_id']}")
            print(f"Room Number : {allocation['room_no']}")
            print(f"Guest Name  : {allocation['guest_name']}")
            print(f"Phone       : {allocation['phone']}")
            print(f"Nights      : {allocation['nights']}")
            print(f"Rate        : ${rate:.2f} per night")
            print(f"Check-in    : {allocation['check_in']}")
            print_rule()

    except KeyError as error:
        print(f"Allocation details could not be displayed. Missing field: {error}")
    except IndexError as error:
        print(f"Allocation list index error: {error}")


def menu_display_allocations():
    """Keyboard-input wrapper for displaying active allocations."""
    display_room_allocation_details()

### Option 6 — Billing and De-Allocation

In [8]:
def billing_and_deallocate_record(room_no):
    """
    Generates a bill and releases the room for the next booking.

    Menu option covered:
        6. Billing and De-Allocation

    Exception handling included:
        ZeroDivisionError is handled in case stored nights are accidentally changed to zero.
    """
    try:
        clean_room_no = normalise_room_number(room_no)
        room = find_room(clean_room_no)
        allocation = find_allocation(clean_room_no)

        if room is None:
            return "Room was not found."

        if allocation is None:
            return "This room is not currently allocated."

        nights = int(allocation["nights"])
        nightly_rate = float(room["nightly_rate"])

        room_charge = nightly_rate * nights
        gst_amount = room_charge * 0.15
        final_total = room_charge + gst_amount
        average_per_night = final_total / nights

        receipt = f"""
LANGHAM HOTEL BILL
Room Number       : {clean_room_no}
Guest Name        : {allocation['guest_name']}
Phone             : {allocation['phone']}
Nights Stayed     : {nights}
Rate per Night    : ${nightly_rate:.2f}
Room Charge       : ${room_charge:.2f}
GST 15%           : ${gst_amount:.2f}
Total Payable     : ${final_total:.2f}
Average per Night : ${average_per_night:.2f}
"""

        room_allocations.remove(allocation)
        room["status"] = "Available"

        return receipt.strip()

    except ZeroDivisionError:
        return "Billing failed because the number of nights is zero."
    except ValueError as error:
        return f"Billing failed. {error}"
    except TypeError as error:
        return f"Billing failed due to incorrect data type: {error}"
    except KeyError as error:
        return f"Billing failed because stored data is missing this field: {error}"


def menu_billing_and_deallocate():
    """Keyboard-input wrapper for billing and de-allocation."""
    room_no = read_required_text("Enter room number for checkout: ")
    print(billing_and_deallocate_record(room_no))

## 4. File I/O Operations

These functions meet the menu requirements for saving, reading, backing up, and clearing the room allocation text file.

### Option 7 — Save Room Allocation to a File

In [9]:
def build_allocation_report_text():
    """Builds a plain text report for the current room allocations."""
    lines = []
    lines.append("LANGHAM HOTEL MANAGEMENT SYSTEM")
    lines.append("ROOM ALLOCATION REPORT")
    lines.append(f"Generated: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
    lines.append("-" * 70)

    if len(room_allocations) == 0:
        lines.append("No active room allocations.")
    else:
        for allocation in room_allocations:
            room = find_room(allocation["room_no"])
            rate = room["nightly_rate"] if room else 0

            lines.append(f"Booking ID : {allocation['booking_id']}")
            lines.append(f"Room No    : {allocation['room_no']}")
            lines.append(f"Guest      : {allocation['guest_name']}")
            lines.append(f"Phone      : {allocation['phone']}")
            lines.append(f"Nights     : {allocation['nights']}")
            lines.append(f"Rate       : ${rate:.2f}")
            lines.append(f"Check-in   : {allocation['check_in']}")
            lines.append("-" * 70)

    return "\n".join(lines)


def save_room_allocation_to_file():
    """
    Saves allocation data to LHMS_StudentID.txt.

    Menu option covered:
        7. Save Room Allocation to a File
    """
    try:
        report_text = build_allocation_report_text()

        with ALLOCATION_FILE.open("w", encoding="utf-8") as file_object:
            file_object.write(report_text)

        return f"Allocation data has been saved to {ALLOCATION_FILE}."

    except IOError as error:
        return f"File saving failed due to an I/O error: {error}"
    except OSError as error:
        return f"File saving failed due to an operating system error: {error}"


def menu_save_allocations():
    """Keyboard-input wrapper for saving allocations."""
    print(save_room_allocation_to_file())

### Option 8 — Display Room Allocation from the File

In [10]:
def display_room_allocation_from_file():
    """
    Reads and displays the contents of the allocation file.

    Menu option covered:
        8. Display Room Allocation from the File
    """
    try:
        with ALLOCATION_FILE.open("r", encoding="utf-8") as file_object:
            content = file_object.read()

        if content.strip() == "":
            print("The allocation file exists, but it is empty.")
        else:
            print("\nContent from allocation file")
            print_rule("=")
            print(content)

    except FileNotFoundError:
        print("The allocation file was not found. Please use Option 7 first.")
    except IOError as error:
        print(f"The allocation file could not be opened: {error}")


def menu_display_file():
    """Keyboard-input wrapper for reading the allocation file."""
    display_room_allocation_from_file()

### Option 9 — Backup and Clear Room Allocation File

In [11]:
def backup_and_clear_room_allocation_file():
    """
    Copies allocation file content into a timestamped backup file, then clears the original file.

    Menu option covered:
        9. Backup and Clear Room Allocation File
    """
    try:
        if not ALLOCATION_FILE.exists():
            return "The allocation file does not exist. Please use Option 7 first."

        content = ALLOCATION_FILE.read_text(encoding="utf-8")

        if content.strip() == "":
            return "The allocation file is already empty, so no backup was created."

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_file = Path(f"LHMS_{STUDENT_ID}_Backup_{timestamp}.txt")

        with backup_file.open("a", encoding="utf-8") as backup_object:
            backup_object.write(content)
            backup_object.write("\n")

        with ALLOCATION_FILE.open("w", encoding="utf-8") as original_file:
            original_file.write("")

        return f"Backup created as {backup_file}. The original allocation file has been cleared."

    except FileNotFoundError:
        return "The allocation file could not be found."
    except IOError as error:
        return f"Backup failed because of a file I/O error: {error}"


def menu_backup_and_clear():
    """Keyboard-input wrapper for backup and clear operation."""
    print(backup_and_clear_room_allocation_file())

## 5. Console Menu

Run the final cell in this section to start the interactive console application.  
The menu uses options **0 to 9**, matching the assessment requirements.

In [16]:
def show_main_menu():
    """Displays the main application menu."""
    print_rule("*")
    print("              LANGHAM HOTEL MANAGEMENT SYSTEM")
    print("                         MAIN MENU")
    print_rule("*")
    print("1. Add Room")
    print("2. Delete Room")
    print("3. Display Room Details")
    print("4. Allocate Room")
    print("5. Display Room Allocation Details")
    print("6. Billing and De-Allocation")
    print("7. Save Room Allocation to a File")
    print("8. Display Room Allocation from the File")
    print("9. Backup and Clear Room Allocation File")
    print("0. Exit Application")
    print_rule("*")


menu_actions = {
    1: menu_add_room,
    2: menu_delete_room,
    3: menu_display_rooms,
    4: menu_allocate_room,
    5: menu_display_allocations,
    6: menu_billing_and_deallocate,
    7: menu_save_allocations,
    8: menu_display_file,
    9: menu_backup_and_clear,
}


def run_hotel_management_system():
    """
    Main program loop for the console application.

    Handles:
        ValueError through read_integer()
        KeyboardInterrupt if the user presses Ctrl+C
        EOFError if input stops unexpectedly
        Exception for any unexpected runtime issue
    """
    while True:
        try:
            show_main_menu()
            choice = read_integer("Enter your choice number: ", minimum=0)

            if choice == 0:
                print("Thank you for using Langham Hotel Management System.")
                break

            selected_action = menu_actions.get(choice)

            if selected_action is None:
                print("Invalid menu option. Please choose a number from 0 to 9.")
            else:
                selected_action()

        except KeyboardInterrupt:
            print("\nProgram stopped by user.")
            break
        except EOFError:
            print("\nInput stream closed. Program stopped safely.")
            break
        except Exception as error:
            print(f"Unexpected application error: {error}")



run_hotel_management_system()

**********************************************************************
              LANGHAM HOTEL MANAGEMENT SYSTEM
                         MAIN MENU
**********************************************************************
1. Add Room
2. Delete Room
3. Display Room Details
4. Allocate Room
5. Display Room Allocation Details
6. Billing and De-Allocation
7. Save Room Allocation to a File
8. Display Room Allocation from the File
9. Backup and Clear Room Allocation File
0. Exit Application
**********************************************************************

Program stopped by user.
Enter your choice number: 2


## 6. Sample Test Run Without Keyboard Input

This cell uses direct function calls instead of `input()`.  
It is useful for checking that the main program logic works before taking screenshots from the interactive menu.

In [13]:
def reset_sample_data():
    """Clears all current data so the test run starts fresh."""
    rooms.clear()
    room_allocations.clear()


reset_sample_data()

print(add_room_record(101, "Standard", 120, "WiFi, TV, Queen bed"))
print(add_room_record(205, "Deluxe", 180, "WiFi, City view, King bed"))
print(add_room_record(310, "Suite", 260, "WiFi, Lounge, Breakfast"))

display_room_details()

print(allocate_room_record(205, "Ariana Smith", "0215557788", 3))
display_room_allocation_details()

print(billing_and_deallocate_record(205))
display_room_allocation_details()

Room 101 has been added successfully.
Room 205 has been added successfully.
Room 310 has been added successfully.

Current Room Details
Room Number : 101
Type        : Standard
Rate        : $120.00 per night
Features    : WiFi, TV, Queen bed
Status      : Available
----------------------------------------------------------------------
Room Number : 205
Type        : Deluxe
Rate        : $180.00 per night
Features    : WiFi, City view, King bed
Status      : Available
----------------------------------------------------------------------
Room Number : 310
Type        : Suite
Rate        : $260.00 per night
Features    : WiFi, Lounge, Breakfast
Status      : Available
----------------------------------------------------------------------
Room 205 has been allocated to Ariana Smith.

Active Room Allocations
Booking ID  : BK-205-20260611025546
Room Number : 205
Guest Name  : Ariana Smith
Phone       : 0215557788
Nights      : 3
Rate        : $180.00 per night
Check-in    : 11/06/2026 02:5

## 7. Error Detection and Corrections

The following list records six practical errors that were considered during development and how the program corrects them.

In [14]:
error_corrections = [
    {
        "error": "ValueError from numeric input",
        "cause": "User may type text when the program expects a number.",
        "fix": "read_integer() and read_decimal() repeat the prompt until valid input is entered."
    },
    {
        "error": "Logical error: duplicate room number",
        "cause": "The same room could be added twice.",
        "fix": "add_room_record() checks find_room() before adding a new room."
    },
    {
        "error": "Logical error: deleting an allocated room",
        "cause": "Deleting an occupied room would leave a booking without a room.",
        "fix": "delete_room_record() blocks deletion when room status is Allocated."
    },
    {
        "error": "Runtime error: missing allocation file",
        "cause": "The user may select display-file before saving the file.",
        "fix": "display_room_allocation_from_file() catches FileNotFoundError."
    },
    {
        "error": "ZeroDivisionError in billing",
        "cause": "Stored booking nights could be corrupted to zero.",
        "fix": "billing_and_deallocate_record() catches ZeroDivisionError."
    },
    {
        "error": "KeyError from incomplete dictionary records",
        "cause": "A room or allocation dictionary could be missing an expected field.",
        "fix": "Display and billing functions catch KeyError and print a clear message."
    }
]

for number, item in enumerate(error_corrections, start=1):
    print(f"{number}. {item['error']}")
    print(f"   Cause: {item['cause']}")
    print(f"   Fix  : {item['fix']}")

1. ValueError from numeric input
   Cause: User may type text when the program expects a number.
   Fix  : read_integer() and read_decimal() repeat the prompt until valid input is entered.
2. Logical error: duplicate room number
   Cause: The same room could be added twice.
   Fix  : add_room_record() checks find_room() before adding a new room.
3. Logical error: deleting an allocated room
   Cause: Deleting an occupied room would leave a booking without a room.
   Fix  : delete_room_record() blocks deletion when room status is Allocated.
4. Runtime error: missing allocation file
   Cause: The user may select display-file before saving the file.
   Fix  : display_room_allocation_from_file() catches FileNotFoundError.
5. ZeroDivisionError in billing
   Cause: Stored booking nights could be corrupted to zero.
   Fix  : billing_and_deallocate_record() catches ZeroDivisionError.
6. KeyError from incomplete dictionary records
   Cause: A room or allocation dictionary could be missing an exp

## 8. Built-in Exception Handling Evidence

The assessment asks for exception handling evidence for common Python exceptions.  
The following cell demonstrates controlled examples for each exception type without breaking the notebook.

In [15]:
def demonstrate_required_exceptions():
    """Demonstrates 11 exception types safely using small controlled examples."""
    print("Exception Handling Demonstration")
    print_rule("=")

    # 1. SyntaxError
    try:
        compile("if True print('missing colon')", "syntax_demo", "exec")
    except SyntaxError as error:
        print("1. SyntaxError       handled:", error.__class__.__name__)

    # 2. ValueError
    try:
        int("abc")
    except ValueError as error:
        print("2. ValueError        handled:", error.__class__.__name__)

    # 3. ZeroDivisionError
    try:
        10 / 0
    except ZeroDivisionError as error:
        print("3. ZeroDivisionError handled:", error.__class__.__name__)

    # 4. IndexError
    try:
        example_list = ["a", "b"]
        example_list[5]
    except IndexError as error:
        print("4. IndexError        handled:", error.__class__.__name__)

    # 5. NameError
    try:
        missing_variable
    except NameError as error:
        print("5. NameError         handled:", error.__class__.__name__)

    # 6. TypeError
    try:
        "Room " + 101
    except TypeError as error:
        print("6. TypeError         handled:", error.__class__.__name__)

    # 7. OverflowError
    try:
        math.exp(99999)
    except OverflowError as error:
        print("7. OverflowError     handled:", error.__class__.__name__)

    # 8. IOError
    try:
        open(Path("."), "r")
    except IOError as error:
        print("8. IOError           handled:", error.__class__.__name__)

    # 9. ImportError
    try:
        from math import function_that_does_not_exist
    except ImportError as error:
        print("9. ImportError       handled:", error.__class__.__name__)

    # 10. EOFError
    try:
        raise EOFError("Simulated input ending early")
    except EOFError as error:
        print("10. EOFError         handled:", error.__class__.__name__)

    # 11. FileNotFoundError
    try:
        open("missing_allocation_file_999.txt", "r")
    except FileNotFoundError as error:
        print("11. FileNotFoundError handled:", error.__class__.__name__)


demonstrate_required_exceptions()

Exception Handling Demonstration
1. SyntaxError       handled: SyntaxError
2. ValueError        handled: ValueError
3. ZeroDivisionError handled: ZeroDivisionError
4. IndexError        handled: IndexError
5. NameError         handled: NameError
6. TypeError         handled: TypeError
7. OverflowError     handled: OverflowError
8. IOError           handled: IsADirectoryError
9. ImportError       handled: ImportError
10. EOFError         handled: EOFError
11. FileNotFoundError handled: FileNotFoundError


## 9. Pseudocode for the Main Program

This pseudocode is kept language-independent and uses clear keywords.

```text
START
    SET room list to empty
    SET allocation list to empty

    WHILE user has not selected Exit
        DISPLAY main menu
        READ user choice

        IF choice is Add Room
            READ room number, room type, rate and features
            IF room number already exists
                WRITE duplicate room message
            ELSE
                ADD room to room list
            ENDIF

        ELSE IF choice is Delete Room
            READ room number
            IF room is allocated
                WRITE cannot delete message
            ELSE
                DELETE room from room list
            ENDIF

        ELSE IF choice is Display Room Details
            WRITE all room details

        ELSE IF choice is Allocate Room
            READ room number and customer details
            IF room is available
                ADD booking to allocation list
                CHANGE room status to allocated
            ELSE
                WRITE room not available message
            ENDIF

        ELSE IF choice is Display Room Allocation Details
            WRITE all active bookings

        ELSE IF choice is Billing and De-Allocation
            READ room number
            CALCULATE bill
            WRITE bill
            REMOVE booking
            CHANGE room status to available

        ELSE IF choice is Save Room Allocation to a File
            WRITE allocation list to text file

        ELSE IF choice is Display Room Allocation from the File
            READ text file
            WRITE file content

        ELSE IF choice is Backup and Clear Room Allocation File
            READ allocation file
            WRITE content to backup file
            CLEAR allocation file

        ELSE IF choice is Exit
            WRITE closing message

        ELSE
            WRITE invalid choice message
        ENDIF
    ENDWHILE
END
```